In [1]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
words = ['anna','bob']

In [3]:
b = {}   #dictionarry to store all the training example

for word in words:
    chs = '.' + word + '.'
    for first, second in zip(chs, chs[1:]):

        if (first, second) not in b:
            b[(first, second)] = 1
        else :
            b[(first, second)] += 1

print(b)

{('.', 'a'): 1, ('a', 'n'): 1, ('n', 'n'): 1, ('n', 'a'): 1, ('a', '.'): 1, ('.', 'b'): 1, ('b', 'o'): 1, ('o', 'b'): 1, ('b', '.'): 1}


In [4]:
map = {
    '.' : 0,
    'a' : 1,
    'b' : 2,
    'n' : 3,
    'o' : 4,
}


In [5]:
mat = torch.zeros((5,5))

print(mat)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [6]:
p = torch.tensor([0.7, 0.2, 0.1])
print(torch.multinomial(p, num_samples=1))

tensor([2])


In [7]:
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))

# We use '.' as the special boundary token.
# It represents both the beginning and end of a name.

stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi['.'] = 0

itos = {i: ch for ch, i in stoi.items()}

print("Vocabulary:")
print(stoi)

Vocabulary:
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [8]:
# ---------------------------------------------------------
# 3. Create the count matrix
# ---------------------------------------------------------

N = torch.zeros((len(stoi), len(stoi)), dtype=torch.int32)


# ---------------------------------------------------------
# 4. Count all character transitions
# ---------------------------------------------------------

for word in words:

    # Add boundary tokens
    chs = ['.'] + list(word) + ['.']

    # Create consecutive character pairs
    for ch1, ch2 in zip(chs, chs[1:]):

        # Convert characters → integer IDs
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        # Increment the corresponding count
        N[ix1, ix2] += 1

In [9]:
# ---------------------------------------------------------
# 5. Convert counts → probabilities
# ---------------------------------------------------------

# Sum each row.
# Each row represents one current character.
row_sums = N.sum(dim=1, keepdim=True)

# Normalize each row.
P = N.float() / row_sums

In [10]:
# ---------------------------------------------------------
# 6. Generate new names
# ---------------------------------------------------------

generator = torch.Generator().manual_seed(42)

num_names = 20

for _ in range(num_names):

    # Start with the boundary token '.'
    ix = 0

    generated_name = []

    while True:

        # Probability distribution for the next character
        p = P[ix]

        # Sample the next character
        ix = torch.multinomial(
            p,
            num_samples=1,
            generator=generator
        ).item()

        # If we generated '.', the name is finished
        if ix == 0:
            break

        # Convert integer ID → character
        generated_name.append(itos[ix])

    print("".join(generated_name))

ya
syahavilin
dleekahmangonya
tryahe
chen
ena
da
amiiae
a
keles
ly
a
oy
asityi
pepolannezale
shahlamion
nacelucyanarivieriaquten
kigshmole
ei
tonylyan


In [11]:


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

words = open("names.txt", "r").read().splitlines()


# --------------------------------------------------
# 2. Vocabulary
# --------------------------------------------------

chars = sorted(list(set("".join(words))))

stoi = {
    '<START>': 0,
    '<END>': 1,
}

for i, ch in enumerate(chars, start=2):
    stoi[ch] = i

itos = {i: ch for ch, i in stoi.items()}


# --------------------------------------------------
# 3. Trigram count tensor
# --------------------------------------------------

N = torch.zeros(
    (len(stoi), len(stoi), len(stoi)),
    dtype=torch.int32
)


# --------------------------------------------------
# 4. Count trigrams
# --------------------------------------------------

for word in words:

    chs = ['<START>', '<START>'] + list(word) + ['<END>']

    for ch1, ch2, ch3 in zip(
        chs,
        chs[1:],
        chs[2:]
    ):

        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]

        N[ix1, ix2, ix3] += 1


# --------------------------------------------------
# 5. Convert counts → probabilities
# --------------------------------------------------

counts = N.sum(dim=2, keepdim=True)

P = N.float() / counts


# --------------------------------------------------
# 6. Generate names
# --------------------------------------------------

g = torch.Generator().manual_seed(42)

for _ in range(20):

    ix1 = stoi['<START>']
    ix2 = stoi['<START>']

    out = []

    while True:

        # P(next | previous, current)
        p = P[ix1, ix2]

        # Sample next character
        ix3 = torch.multinomial(
            p,
            num_samples=1,
            generator=g
        ).item()

        # End of name
        if ix3 == stoi['<END>']:
            break

        # Add character
        out.append(itos[ix3])

        # Shift context
        ix1 = ix2
        ix2 = ix3

    print("".join(out))

kannlee
leen
maksanshikaevicamaka
va
bleigda
deko
lihadanajerlayani
sa
lin
stilotonna
mey
jen
ruzeelleen
abdomie
nelluwan
taes
marcartli
ludhanney
bren
sriya


In [12]:
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi['.'] = 0

itos = {i: ch for ch, i in stoi.items()}

N = torch.zeros((len(stoi), len(stoi)), dtype=torch.int32)

for word in words:

    # Add boundary tokens
    chs = ['.'] + list(word) + ['.']

    # Create consecutive character pairs
    for ch1, ch2 in zip(chs, chs[1:]):

        # Convert characters → integer IDs
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        # Increment the corresponding count
        N[ix1, ix2] += 1

row_sums = N.sum(dim=1, keepdim=True)

# Normalize each row.
P = N.float() / row_sums


In [13]:
log_likely_hood = 0
n = 0

for word in words[:3]:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        # print(ch1, ch2)
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likely_hood += -logprob
        n += 1
        print(-logprob)

print(f"Total loss of current setup = {log_likely_hood/n}")

tensor(3.0408)
tensor(3.2793)
tensor(3.6772)
tensor(0.9418)
tensor(1.6299)
tensor(4.3982)
tensor(2.5508)
tensor(1.7278)
tensor(4.1867)
tensor(1.0383)
tensor(1.9796)
tensor(1.6299)
tensor(1.9829)
tensor(3.7045)
tensor(1.3882)
tensor(1.6299)
Total loss of current setup = 2.424102306365967


In [14]:
# creating training sets for bigrams

xs = []
ys = []

for w in words[:1]:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print('xs =',xs)
print('ys =',ys)

. a
a v
v a
a .
xs = tensor([ 0,  1, 22,  1])
ys = tensor([ 1, 22,  1,  0])


In [33]:
import torch.nn.functional as F
import matplotlib.pyplot as plt


xenc = F.one_hot(xs, num_classes=27).float()
print(xenc, xenc.shape, xenc.dtype)

# plt.imshow(xenc)

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]]) torch.Size([4, 27]) torch.float32


In [68]:
g = torch.Generator().manual_seed(214748644)
w = torch.randn((27,27), generator=g)
w
logits = (xenc @ w)
counts = logits.exp()    #softmax implementation
probs = counts/ counts.sum(1, keepdims = True)
probs

tensor([[0.0220, 0.0070, 0.0173, 0.0101, 0.0108, 0.0088, 0.0172, 0.0214, 0.0129,
         0.0017, 0.0097, 0.0082, 0.2376, 0.0147, 0.0265, 0.0845, 0.0209, 0.0100,
         0.0890, 0.0210, 0.0701, 0.0188, 0.0329, 0.0350, 0.0442, 0.0154, 0.1322],
        [0.0131, 0.0460, 0.0136, 0.0111, 0.0309, 0.0315, 0.0561, 0.0294, 0.0525,
         0.0614, 0.0085, 0.0092, 0.1384, 0.0620, 0.0207, 0.0346, 0.1050, 0.0393,
         0.0225, 0.1078, 0.0051, 0.0203, 0.0263, 0.0066, 0.0206, 0.0081, 0.0193],
        [0.0212, 0.0034, 0.0033, 0.0057, 0.0272, 0.0065, 0.0065, 0.0043, 0.0039,
         0.0120, 0.0292, 0.2617, 0.0292, 0.0353, 0.2750, 0.0156, 0.0145, 0.0818,
         0.0248, 0.0153, 0.0089, 0.0068, 0.0330, 0.0096, 0.0391, 0.0160, 0.0100],
        [0.0131, 0.0460, 0.0136, 0.0111, 0.0309, 0.0315, 0.0561, 0.0294, 0.0525,
         0.0614, 0.0085, 0.0092, 0.1384, 0.0620, 0.0207, 0.0346, 0.1050, 0.0393,
         0.0225, 0.1078, 0.0051, 0.0203, 0.0263, 0.0066, 0.0206, 0.0081, 0.0193]])

In [50]:
probs.shape

torch.Size([4, 27])

In [51]:
probs[0].sum()

tensor(1.0000)

In [69]:
nlls = torch.zeros(4)
for i in range(4):
    x = xs[i].item()
    y = ys[i].item()
    p = probs[i,y]

    logp = torch.log(p)
    nll = -logp
    nlls[i] = nll

nlls[i].mean().item()    # total loss

4.334928035736084

In [91]:
# randomly generated 27 neurons weight, each neuron receives 27 input
g = torch.Generator().manual_seed(214748644)
w = torch.randn((27,27), generator=g, requires_grad = True)

In [102]:
#forward pass
logits = (xenc @ w)
counts = logits.exp()    #softmax implementation
probs = counts/ counts.sum(1, keepdims = True)
loss = -probs[torch.arange(4), ys].log().mean()

In [103]:
loss

tensor(4.5962, grad_fn=<NegBackward0>)

In [100]:
#backward pass
w.grad = None
loss.backward()

In [101]:
w.data +=  -0.1 * w.grad